This notebook analyzes interaction frequencies for selected simulations and computes similarity and clustering metrics.

This notebook is organised into clear sections: imports, setup, helper functions, analysis, plotting, and saving results.


##### 1. Import necessary libraries

In [1]:
import MDAnalysis, MDAnalysisTests


In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
import MDAnalysis as mda
from MDAnalysis.analysis import align, rms
from MDAnalysis.analysis.rms import RMSF
from scipy.spatial.distance import jaccard, pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram, cut_tree
from sklearn.metrics import pairwise_distances, classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier, export_graphviz, plot_tree
from sklearn.model_selection import train_test_split
from io import StringIO


##### 2. Mount drive with necessary data

In [3]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##### 3. Setup the environment directory

In [4]:
rootdir = '/content/drive/MyDrive/M1_STAGE/Data/interactions/'
os.chdir(rootdir)

##### 4. Define helpful functions

In [5]:
def find_hbonds(df: pd.DataFrame) -> list[str]:
    """Return column names that correspond to hydrogen bonds (contain 'hb')."""
    return [col for col in df.columns if 'hb' in col]


def phase_frequency(df_phase: pd.DataFrame, hbond_cols: list[str]) -> dict:
    """
    Compute the occurrence frequency (%) of each H-bond in a phase DataFrame.
    Returns 0.0 for empty phases or missing columns.
    """
    n = len(df_phase)
    freqs = {}
    for col in hbond_cols:
        if col in df_phase.columns and n > 0:
            freqs[col] = (df_phase[col] > 0).sum() / n * 100
    return freqs


def slice_transition_state(df_full: pd.DataFrame, sim_name: str) -> pd.DataFrame:
    """Return only the transition-state rows defined in SELECTED_FRAMES."""
    if sim_name not in SELECTED_FRAMES:
        print(f"[WARN] No frame range for {sim_name}. Using full DataFrame.")
        return df_full
    valid = [i for i in df_full.index if i in SELECTED_FRAMES[sim_name]]
    return df_full.loc[valid]


def get_residue_numbers(bond_name: str) -> tuple[int, int]:
    """
    Parse a bond column name and return the two residue numbers involved.
    Expected format: ..._RES1_<num1>_..._RES2_<num2>_...
    Falls back to extracting all integers and taking the first two.
    Returns (None, None) if parsing fails.
    """
    import re
    nums = [int(x) for x in re.findall(r'\d+', bond_name)]
    if len(nums) >= 2:
        return nums[0], nums[1]
    return None, None


def classify_chain(resnum: int) -> str | None:
    """Return 'A', 'B', or None based on residue number."""
    if resnum is None:
        return None
    if 1 <= resnum <= 99:
        return 'A'
    if 100 <= resnum <= 198:
        return 'B'
    return None


def is_interchain(bond_name: str) -> bool:
    """Return True if the bond connects one residue in chain A and one in chain B."""
    r1, r2 = get_residue_numbers(bond_name)
    return classify_chain(r1) != classify_chain(r2) and None not in (
        classify_chain(r1), classify_chain(r2))


def high_variance_bonds(freq_table: pd.DataFrame,
                        min_delta: float = 20.0,
                        top_n: int | None = None) -> pd.DataFrame:
    """
    Keep bonds whose frequency changes substantially between at least one ph1/ph2 pair.

    Parameters
    ----------
    freq_table : DataFrame  (bonds × phases)
    min_delta  : minimum |ph2 - ph1| in % for at least one simulation
    top_n      : if set, keep only the top_n bonds by max |delta| across sims
    """
    sim_names = list({c.rsplit('_', 1)[0] for c in freq_table.columns})
    max_delta = pd.Series(0.0, index=freq_table.index)

    for sim in sim_names:
        ph1 = f"{sim}_ph1"
        ph2 = f"{sim}_ph2"
        if ph1 in freq_table.columns and ph2 in freq_table.columns:
            delta = (freq_table[ph2] - freq_table[ph1]).abs()
            max_delta = max_delta.combine(delta, max)

    mask = max_delta >= min_delta
    result = freq_table[mask]
    if top_n is not None:
        result = result.loc[max_delta[mask].nlargest(top_n).index]
    return result


def split_phases(df_ts: pd.DataFrame, sim_name: str):
    """
    Split transition-state DataFrame into phase 1 and phase 2.
    Returns (df_ph1, df_ph2) or raises if no phase limit is defined.
    """
    if sim_name not in PHASE_LIMITS:
        raise ValueError(f"No phase limit defined for {sim_name}.")
    lim = PHASE_LIMITS[sim_name]
    idx = df_ts.index
    ph1 = df_ts.loc[idx[idx <= lim]] if (idx <= lim).any() else pd.DataFrame(columns=df_ts.columns)
    ph2 = df_ts.loc[idx[idx >  lim]] if (idx >  lim).any() else pd.DataFrame(columns=df_ts.columns)
    return ph1, ph2

##### 5. Setup the constants

In [6]:
OUTPUT_DIR = '/content/drive/MyDrive/M1_STAGE/Manips/Tables/'     # adapt to your env
INTERACTION_FILES = ['res_V12.csv', 'res_V7.csv']
SIMULATION_NAMES  = ['V12', 'V7']

# Frames of interest per simulation (transition-state window)
SELECTED_FRAMES = {
    'V12': range(0,   71 + 1),
    'V7':  range(163,  234 + 1),
    # 'V1':  range(412, 587 + 1),
    # 'V8':  range(62,  237 + 1),
    # 'V21': range(62,  237 + 1),
    # 'V11': range(698, 873 + 1),
}

# Index threshold separating phase 1 from phase 2
PHASE_LIMITS = {
    'V12': 35,
    'V7':  198,
    # 'V1':  517,
    # 'V8':  180,
    # 'V21': 155,
    # 'V11': 1000,
}
# Desired column order in the frequency table
PHASES_ORDERED = ['V12_ph1', 'V12_ph2', 'V7_ph1', 'V7_ph2']

In [7]:
all_phase_frequencies = {}
df_V7_V12 = pd.DataFrame()
for sim_file, sim_name in zip(INTERACTION_FILES, SIMULATION_NAMES):
    df_full = pd.read_csv(sim_file)
    #df_ts     = slice_transition_state(df_full, sim_name)
    #df_ph1, df_ph2 = split_phases(df_ts, sim_name)
    hbonds = find_hbonds(df_full)
    #globals()[f"{sim_name}_df"] = pd.concat([df_ph1, df_ph2])
    globals()[f"{sim_name}_df"] = df_full

    #print(f"{sim_name}: {len(df_ph1)} frames in ph1, {len(df_ph2)} frames in ph2, "
    #      f"{len(globals()[f'{sim_name}_df'].columns)} H-bonds found.")
    print(f"{sim_name}: {len(globals()[f'{sim_name}_df'])} frames, "
          f"{len(hbonds)} H-bonds found.")

V12: 1001 frames, 1741 H-bonds found.
V7: 1001 frames, 1741 H-bonds found.


In [8]:
df_V7_V12 = pd.concat([df_V7_V12, V7_df, V12_df])
df_V7_V12.tail()

,vdw_ALA_172_C_VAL_161_O,vdw_GLY_49_CA_MET_76_CE,vdw_GLY_52_O_ILE_149_N,vdw_ILE_114_CA_ILE_163_CD,vdw_GLY_185_C_LEU_189_CD1,vdw_ARG_186_CB_THR_190_CG2,vdw_ILE_181_CB_THR_179_CG2,vdw_GLY_17_C_TYR_14_CA,vdw_ILE_32_CG2_ILE_82_CB,vdw_ARG_186_CB_TRP_6_NE1,...,hbsb_ALA_133_N_ASN_182_OD1,vdw_ALA_127_C_ARG_186_CG,hbsb_PRO_1_O_SER_4_OG,vdw_ILE_149_O_PRO_180_CB,vdw_GLN_117_CD_GLY_134_O,vdw_LEU_5_CB_PHE_3_CD1,vdw_GLU_162_CD_GLU_164_OE1,vdw_GLY_51_C_PHE_152_C,vdw_ASN_98_CG_PRO_1_O,hbsb_GLN_2_NE2_SER_4_N
996,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
997,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
998,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
999,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
1000,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0


In [9]:
df_frequency = pd.DataFrame()

for sim_name in SIMULATION_NAMES:
    # Filter out the columns that are either never present or always present in both phases and remove
    df = df.loc[:, df.sum(axis=0) > 0]
    df = df.loc[:, df.sum(axis=0) != len(df.index)]
    globals()[f"{sim_name}_df_ts"] = df.loc[df.index.isin(SELECTED_FRAMES[f'{sim_name}'])]
    #split back into phases
    df_ph1, df_ph2 = split_phases(globals()[f"{sim_name}_df_ts"], sim_name)

    #define ph1 and ph2 frequencies
    globals()[f"{sim_name}_df_ph1_freq"] = phase_frequency(df_ph1, hbonds)
    globals()[f"{sim_name}_df_ph2_freq"] = phase_frequency(df_ph2, hbonds)
    
    df_frequency[f"{sim_name}_ph1"] = pd.DataFrame.from_dict(globals()[f"{sim_name}_df_ph1_freq"], orient='index')
    df_frequency[f"{sim_name}_ph2"] = pd.DataFrame.from_dict(globals()[f"{sim_name}_df_ph2_freq"], orient='index')
    df_frequency[f"{sim_name}_delta"] = (df_frequency[f"{sim_name}_ph1"] - df_frequency[f"{sim_name}_ph2"]).abs()

NameError: name 'df' is not defined

In [ ]:
df_delta = df_frequency[[col for col in df_frequency.columns if col.endswith('_delta')]]
#remove cols with delta below 20%
MIN_DELTA_PCT = 20
df_delta_filtered = df_delta[df_delta['V12_delta'] >= MIN_DELTA_PCT]
df_delta_filtered = df_delta[df_delta['V7_delta'] >= MIN_DELTA_PCT]
interactions_filtered = df_delta_filtered.index

interactions_list = list(interactions_filtered)

V7_df_ts = V7_df_ts[interactions_filtered]
V12_df_ts = V12_df_ts[interactions_filtered]

V7_df = V7_df[interactions_list]
V12_df = V12_df[interactions_list]

In [ ]:
print(f'Interactions after filtering: V7 = {len(V7_df.columns)}, V12 = {len(V12_df.columns)}')

In [ ]:
arr_V7 = V7_df.to_numpy()
arr_V12 = V12_df.to_numpy()

jaccard_sim_V7_V7 = pairwise_distances(arr_V7,arr_V7, metric='jaccard', n_jobs=-1)
jaccard_sim_V7_V12 = pairwise_distances(arr_V7,arr_V12, metric='jaccard', n_jobs=-1)
jaccard_sim_V12_V7 = pairwise_distances(arr_V12,arr_V7, metric='jaccard', n_jobs=-1)
jaccard_sim_V12_V12 = pairwise_distances(arr_V12,arr_V12, metric='jaccard', n_jobs=-1)

In [ ]:
jaccard_df_V7_V7 = pd.DataFrame(jaccard_sim_V7_V7)
jaccard_df_V7_V12 = pd.DataFrame(jaccard_sim_V7_V12)
jaccard_df_V12_V7 = pd.DataFrame(jaccard_sim_V12_V7)
jaccard_df_V12_V12 = pd.DataFrame(jaccard_sim_V12_V12)

jaccard_df_left = pd.DataFrame()
jaccard_df_left = pd.concat([jaccard_df_V7_V7, jaccard_df_V12_V7], ignore_index=True)

jaccard_df_right = pd.DataFrame()
jaccard_df_right = pd.concat([jaccard_df_V7_V12, jaccard_df_V12_V12], ignore_index=True)

jaccard_df_full = pd.DataFrame()
jaccard_df_full = pd.concat([jaccard_df_left, jaccard_df_right], axis=1, ignore_index=True)

In [ ]:
jaccard_df_full.fillna(0, inplace=True)

In [ ]:
def make_space_above(fig, topmargin=1):
    """ increase figure size to make topmargin (in inches) space for 
        titles, without changing the axes sizes"""
    s = fig.subplotpars
    w, h = fig.get_size_inches()

    figh = h - (1-s.top)*h  + topmargin
    fig.subplots_adjust(bottom=s.bottom*h/figh, top=1-topmargin/figh)
    fig.set_figheight(figh) 

In [ ]:
font = {'family': 'sans-serif',
        'style':'normal',
        'color':  'black',
        'fontweight': 'bold',
        'fontsize': 12,
        }
font1 = {'family': 'sans-serif',
        'style': 'oblique',
        'color':  'black',
        'fontweight': 'normal',
        'fontsize': 8,
        }

In [ ]:
#plot jaccard sililarity heatmap
import seaborn as sns
import matplotlib.pyplot as plt
fig = plt.figure(figsize=(10, 8))
plt.imshow(jaccard_df_full, cmap='viridis', origin='upper')

plt.suptitle("Similarité de Jaccard entre les profils \nd'interaction V12 et V7 (dans l'état de transition)", fontdict=font, x=0.5, y=1)
plt.title('\n*filtrage des interactions à haute\nvariance (>20%) entre les phases', fontdict=font1, x=0.9, y=1.02)

plt.axvline(x=1001, color='black', linewidth=1.5, alpha=0.7)
plt.axhline(y=1001, color='black', linewidth=1.5, alpha=0.7)

tick_positions = [500, 1502]
tick_labels = ['V7 (MP)\nFrames 0-1001', 'V12 (DP)\nFrames 0-1001']

plt.xticks(tick_positions, tick_labels, fontsize=10)
plt.yticks(tick_positions, tick_labels, fontsize=10, rotation=90, va='center')

#plt.xlabel('Simulations comparées (Axe X)', fontsize=12, fontweight='bold', labelpad=10)
#plt.ylabel('Simulations comparées (Axe Y)', fontsize=12, fontweight='bold', labelpad=15)
plt.colorbar(label=r'RMSD ($\AA$)')

plt.tight_layout()
#make_space_above(fig, topmargin=1)  
plt.show()


In [ ]:
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, dendrogram, cut_tree
import seaborn as sns

complete_clustering = linkage(jaccard_df_full, method="ward")

In [ ]:
# 1. Define color palette
color_map = {
    'V7_Phase1':  '#FF9F43',  # Bright Orange
    'V7_Phase2':  '#EE5253',  # Crimson Red
    'V12_Phase1': '#0ABDE3',  # Cyan Blue
    'V12_Phase2': '#10AC84'   # Emerald Green
}

# 2. Plot the dendrogram
fig, ax = plt.subplots(figsize=(15, 8))
dendrogram(complete_clustering, leaf_font_size=8, orientation='top')

plt.title("Regroupement de frames individuels de V7 et V12 \nselon distance de Jaccard (simulations complètes, \nclustering par méthode de Ward)",
          fontsize=14, fontweight='bold', pad=15)
plt.ylabel("Distance de Jaccard", fontsize=11)

# 3. Dynamic leaf text color assignment loop
xlbls = ax.get_xmajorticklabels()

for lbl in xlbls:
    text = lbl.get_text()
    if text.isdigit():
        abs_idx = int(text) # This is the absolute index from 0 to 2001
        
        # Determine exactly which simulation and phase this leaf represents
        if abs_idx < 1001:
            sim = 'V7'
            phase = 'Phase1' if abs_idx <= 198 else 'Phase2'
        else:
            sim = 'V12'
            local_frame = abs_idx - 1001
            phase = 'Phase1' if local_frame <= 35 else 'Phase2'
            
        # Select the target color key
        state_key = f"{sim}_{phase}"
        lbl.set_color(color_map[state_key])

# 4. Add a visual legend to the plot so your report reader knows what the colors mean
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=color_map['V7_Phase1'], label='V7 (MP) - Phase 1'),
    Patch(facecolor=color_map['V7_Phase2'], label='V7 (MP) - Phase 2'),
    Patch(facecolor=color_map['V12_Phase1'], label='V12 (DP) - Phase 1'),
    Patch(facecolor=color_map['V12_Phase2'], label='V12 (DP) - Phase 2'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=10, frameon=True)

plt.tight_layout()
plt.show()

In [ ]:

# 1. Define your 4-state color map for the row/column sidebars
color_map = {
    'V7_Phase1':  '#FF9F43',  
    'V7_Phase2':  '#EE5253',  
    'V12_Phase1': '#0ABDE3',  
    'V12_Phase2': '#10AC84'   
}

# 2. Create a list of colors corresponding to every row/column in your 2002 dataset
# This adds a color bar between the tree and the heatmap to label phases explicitly!
meta_colors = []
for abs_idx in range(len(jaccard_df_full)):
    if abs_idx < 1001:
        phase = 'Phase1' if abs_idx <= 198 else 'Phase2'
        meta_colors.append(color_map[f"V7_{phase}"])
    else:
        local_frame = abs_idx - 1001
        phase = 'Phase1' if local_frame <= 35 else 'Phase2'
        meta_colors.append(color_map[f"V12_{phase}"])

# 3. Plot the combined Clustermap
# We pass your pre-computed 'complete_clustering' linkage to both row and column links
g = sns.clustermap(
    jaccard_df_full,
    row_linkage=complete_clustering,
    col_linkage=complete_clustering,
    cmap='viridis',
    row_colors=meta_colors,      # Displays structural phase color track on the left
    col_colors=meta_colors,      # Displays structural phase color track on the top
    figsize=(12, 12),
    xticklabels=False,           # Hide 2002 text labels to avoid overcrowding
    yticklabels=False,
    cbar_kws={'label': 'Distance Jaccard'}
)

# 4. Add titles and adjust the position of the layout elements
g.fig.suptitle("Matrice de Distances Réordonnée par Classification Hiérarchique", 
               fontsize=14, fontweight='bold', y=1.02)

# 5. Add a legend so your professor understands the side colors
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=color_map['V7_Phase1'], label='V7 (MP) - Phase 1'),
    Patch(facecolor=color_map['V7_Phase2'], label='V7 (MP) - Phase 2'),
    Patch(facecolor=color_map['V12_Phase1'], label='V12 (DP) - Phase 1'),
    Patch(facecolor=color_map['V12_Phase2'], label='V12 (DP) - Phase 2'),
]
plt.gca().legend(handles=legend_elements, bbox_to_anchor=(1.7, 1.5), loc='upper right', frameon=True)

plt.show()

In [ ]:
v7_transition_indices = np.arange(198 - 35, 198 + 37)     # 162 to 234 inclusive
v12_transition_indices = np.arange(1036 - 35, 1036 + 37)  # 1000 to 1072 inclusive

target_indices = np.concatenate([v7_transition_indices, v12_transition_indices])

sub_matrix_transition = jaccard_df_full.iloc[target_indices, target_indices]

print("Transition state matrix shape:", sub_matrix_transition.shape) 

complete_clustering = linkage(sub_matrix_transition, method="ward")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram

# 1. Create a lookup list of your true absolute indices (from the 0-2001 system)
# This perfectly preserves the map between the 0-145 layout and your 2002 framework
true_absolute_lookup = list(sub_matrix_transition.index)

# 2. Plot the dendrogram
fig, ax = plt.subplots(figsize=(15, 8))
dendrogram(complete_clustering, leaf_font_size=6, orientation='top')

plt.title("Regroupement de frames individuels de V7 et V12 \nselon distance de Jaccard (fenêtre de transition $\pm$36 ns, \nclustering par méthode de Ward)",
          fontsize=14, fontweight='bold', pad=15)
plt.ylabel("Distance de Jaccard", fontsize=11)

# 3. Dynamic leaf text and color reconstruction loop
xlbls = ax.get_xmajorticklabels()

for lbl in xlbls:
    text = lbl.get_text()
    if text.isdigit():
        # This position_idx is a value from 0 to 145
        position_idx = int(text)
        
        # Pull the true absolute index (0-2001 scale) using our lookup list
        abs_idx = true_absolute_lookup[position_idx]
        
        # Determine simulation and local frame based on the true absolute index
        if abs_idx < 1001:
            sim = 'V7'
            local_frame = abs_idx  # V7 keeps its absolute frame number
            phase = 'Phase1' if local_frame <= 198 else 'Phase2'
        else:
            sim = 'V12'
            local_frame = abs_idx - 1001  # V12 shifts back to 0-1000 range
            phase = 'Phase1' if local_frame <= 35 else 'Phase2'
            
        # Update the text displayed under the leaf to show the actual local frame number
        lbl.set_text(f"{local_frame}")
        
        # Color the leaf according to its true simulation-phase assignment
        state_key = f"{sim}_{phase}"
        lbl.set_color(color_map[state_key])

# 4. Re-apply the changes to the axis object (Matplotlib requirement when modifying text values)
ax.set_xticklabels(xlbls)

# 5. Add Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=color_map['V7_Phase1'], label='V7 (MP) - Phase 1'),
    Patch(facecolor=color_map['V7_Phase2'], label='V7 (MP) - Phase 2'),
    Patch(facecolor=color_map['V12_Phase1'], label='V12 (DP) - Phase 1'),
    Patch(facecolor=color_map['V12_Phase2'], label='V12 (DP) - Phase 2'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=10, frameon=True)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# 1. Define your 4-state color map for the row/column sidebars
color_map = {
    'V7_Phase1':  '#FF9F43',  # Bright Orange
    'V7_Phase2':  '#EE5253',  # Crimson Red
    'V12_Phase1': '#0ABDE3',  # Cyan Blue
    'V12_Phase2': '#10AC84'   # Emerald Green
}

# 2. Build the exact 146-element color tracking bar using your lookup array
# This tells Seaborn exactly what phase each of the 146 sliced lines represents!
true_absolute_lookup = list(sub_matrix_transition.index)
meta_colors = []

for abs_idx in true_absolute_lookup:
    if abs_idx < 1001:
        sim = 'V7'
        local_frame = abs_idx
        phase = 'Phase1' if local_frame <= 198 else 'Phase2'
    else:
        sim = 'V12'
        local_frame = abs_idx - 1001
        phase = 'Phase1' if local_frame <= 35 else 'Phase2'
        
    # Append the matching color string to our tracking list
    meta_colors.append(color_map[f"{sim}_{phase}"])

# 3. Plot the combined Clustermap for the 146x146 transition window
# Note: Ensure complete_clustering was calculated directly on sub_matrix_transition
g = sns.clustermap(
    sub_matrix_transition,
    row_linkage=complete_clustering,
    col_linkage=complete_clustering,
    cmap='viridis',
    row_colors=meta_colors,      # Displays structural phase color track on the left
    col_colors=meta_colors,      # Displays structural phase color track on the top
    figsize=(10, 10),
    xticklabels=False,           # Hidden because 146 text overlapping looks muddy
    yticklabels=False,
    cbar_kws={'label': 'Distance Jaccard'}
)

# 4. Refine layout positions (Seaborn clustermaps can displace titles)
g.fig.suptitle("Matrice Jaccard Réordonnée par Classification Hiérarchique\n(Fenêtre de Transition $\pm$36 ns)", 
               fontsize=13, fontweight='bold', y=1.02)

# 5. Add a clean, un-skewed legend onto the figure canvas
legend_elements = [
    Patch(facecolor=color_map['V7_Phase1'], label='V7 (MP) - Phase 1 (Transition-Avant)'),
    Patch(facecolor=color_map['V7_Phase2'], label='V7 (MP) - Phase 2 (Transition-Après)'),
    Patch(facecolor=color_map['V12_Phase1'], label='V12 (DP) - Phase 1 (Transition-Avant)'),
    Patch(facecolor=color_map['V12_Phase2'], label='V12 (DP) - Phase 2 (Transition-Après)'),
]

# Adding legend to the root figure context so it doesn't get clipped or squished
g.fig.legend(handles=legend_elements, bbox_to_anchor=(1.05, 0.85), loc='upper left', frameon=True)

plt.show()

In [ ]:
#save all calculated matrices as csv files for later use
jaccard_df_V7_V7.to_csv("/content/drive/MyDrive/M1_STAGE/Manips/Tables/jaccard_V7_V7.csv", index=False)
jaccard_df_V7_V12.to_csv("/content/drive/MyDrive/M1_STAGE/Manips/Tables/jaccard_V7_V12.csv", index=False)
jaccard_df_V12_V7.to_csv("/content/drive/MyDrive/M1_STAGE/Manips/Tables/jaccard_V12_V7.csv", index=False)
jaccard_df_V12_V12.to_csv("/content/drive/MyDrive/M1_STAGE/Manips/Tables/jaccard_V12_V12.csv", index=False)
jaccard_df_full.to_csv("/content/drive/MyDrive/M1_STAGE/Manips/Tables/jaccard_full.csv", index=False)
sub_matrix_transition.to_csv("/content/drive/MyDrive/M1_STAGE/Manips/Tables/jaccard_transition_window.csv", index=False)


# Decision Tree

In [ ]:
df_full = pd.DataFrame()
df_full = pd.concat([V7_df, V12_df])

# phase label
limit_V7 = PHASE_LIMITS['V7']
limit_V12 = PHASE_LIMITS['V12']
label = []
for frame in list(df_full.index):
    if frame in range(0, limit_V12 + 1):
        label.append(f"Phase 1")
    elif frame in range(limit_V7 - 35, limit_V7 + 1):
        label.append(f"Phase 1")
    else:
        label.append(f"Phase 2")
        
#df_full["Label"] = label

In [ ]:
#Decision tree to predict phase based on interactions
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
V7_df['Phase'] = ['Phase 1' if f <= limit_V7 else 'Phase 2' for f in V7_df.index]
V12_df['Phase'] = ['Phase 1' if f <= limit_V12 else 'Phase 2' for f in V12_df.index]

df_full = pd.concat([V7_df, V12_df], ignore_index=True)

X = df_full.drop(columns=['Phase'])
y = df_full['Phase'].tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

clf = DecisionTreeClassifier(max_depth=3)
clf.fit(X_train, y_train)

In [ ]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_graphviz, plot_tree
from io import StringIO

fig, ax = plt.subplots(figsize=(8, 4))
plot_tree(clf, 
          feature_names=X.columns.tolist(),
          class_names=clf.classes_.tolist(),
          filled=True, 
          rounded=True,
          proportion=False,
          ax=ax)

#ax.set_title('Decision Tree – discriminating bonds between sim-phases', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'decision_tree.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# check if asp25-asp124 is still in the df
resid_list = ['26','125']
asp_check = [col for col in df_full.columns if all(resid in col for resid in resid_list)]

In [ ]:
asp_check